# ConfigLoader Example

This notebook reads backend-specific example files from `docs/examples/system/<example_kind>/{config,params}` and loads them with `ConfigLoader`.


In [ ]:
from pathlib import Path

import yaml

from qubex.system import ConfigLoader

## Resolve example paths

Choose `example_kind = "quel1"` or `"quel3"` and the notebook resolves the matching example directory.

This supports running the notebook from the repository root, `docs/examples/system`, or one of its child directories.


In [ ]:
example_kind = "quel3"
if example_kind not in {"quel1", "quel3"}:
    raise ValueError(f"Unsupported example_kind: {example_kind}")

candidate_root_dirs = [
    Path.cwd(),
    Path.cwd() / "docs/examples/system",
    Path.cwd().parent,
]

example_root = next(
    (
        path
        for path in candidate_root_dirs
        if (path / "quel1" / "config").is_dir()
        and (path / "quel3" / "config").is_dir()
    ),
    None,
)
if example_root is None:
    raise FileNotFoundError(
        "Could not find `docs/examples/system/{quel1,quel3}` from the current working directory."
    )

example_dir = example_root / example_kind
config_dir = example_dir / "config"
params_dir = example_dir / "params"

print(f"example_root: {example_root.resolve()}")
print(f"example_kind: {example_kind}")
print(f"config_dir  : {config_dir.resolve()}")
print(f"params_dir  : {params_dir.resolve()}")


## Read config files directly


In [ ]:
config_payloads = {}
for path in sorted(config_dir.glob("*.yaml")):
    with path.open() as f:
        config_payloads[path.name] = yaml.safe_load(f)

print("loaded files:", ", ".join(config_payloads))
print("system backend:", config_payloads["system.yaml"]["backend"])
print("chip id:", config_payloads["system.yaml"]["chip_id"])

## Load the same files with `ConfigLoader`


In [ ]:
loader = ConfigLoader(
    chip_id="64Q",
    config_dir=config_dir,
    params_dir=params_dir,
    autoload=False,
)
loader.load()
system = loader.get_experiment_system()

print("backend_kind :", loader.backend_kind)
print("wiring_file  :", loader.wiring_file)
print("qubits       :", len(system.qubits))
print("resonators   :", len(system.resonators))
print("boxes        :", len(system.boxes))

## Access loaded parameters


In [ ]:
print("Q00 control amplitude :", system.control_params.get_control_amplitude("Q00"))
print("Q00 readout amplitude :", system.control_params.get_readout_amplitude("Q00"))
print("MUX0 capture delay    :", system.control_params.get_capture_delay(0))